In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
from einops import repeat
from typing import Optional

In [2]:
batch_size = 128
embedding_dim = 256
patch_size = 16
in_channels = 3
image_height = 512
image_width = 512
num_patches = image_height * image_width // patch_size**2
num_cls_tokens = 1
sequence_length = num_patches + num_cls_tokens
device = "cuda" if torch.cuda.is_available() else "cpu"
num_heads = 32
dropout=0.1
print(device)

cuda


In [3]:
image = torch.randn((batch_size, in_channels, image_height, image_width)).to(device)
image.shape

torch.Size([128, 3, 512, 512])

# Image Embedder
- Transforms image into patch embeddings + adds a learned CLS token

In [4]:
class ImageEmbedder(nn.Module):
    def __init__(self, in_channels: int, embedding_dim: int, patch_size: int):
        super().__init__()
        self.nb_cls_tokens = 1
        self.cls_token = nn.Parameter(torch.randn(1, self.nb_cls_tokens, embedding_dim))
        self.patch_embedder = nn.Conv2d(in_channels=in_channels, out_channels=embedding_dim, kernel_size=patch_size, stride=patch_size, padding=0)
        
    def forward(self, image: torch.Tensor) -> torch.Tensor:
        patch_embeddings = self.patch_embedder(image)
        patch_embeddings = rearrange(patch_embeddings, "b d h_p w_p -> b (h_p w_p) d")
        
        batch_size, _, _ = patch_embeddings.shape

        cls_tokens = repeat(self.cls_token, '1 n_cls d -> b n_cls d', n_cls=self.nb_cls_tokens, b=batch_size)

        embeddings = torch.cat([cls_tokens, patch_embeddings], dim=1)

        return embeddings

In [5]:
image_embedder = ImageEmbedder(in_channels=in_channels,
                               embedding_dim=embedding_dim,
                               patch_size=patch_size).to(device)

In [6]:
embeddings = image_embedder(image)
embeddings.shape

torch.Size([128, 1025, 256])

# Learned 1D Positional Encoding
- Adds positional information to the patch embeddings + cls token

In [7]:
class Learned1dPositionEncoder(nn.Module):
    def __init__(self, sequence_length: int, embedding_dim: int):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, sequence_length, embedding_dim))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pos_embedding

In [8]:
pos_encoder = Learned1dPositionEncoder(sequence_length=sequence_length, embedding_dim=embedding_dim).to(device)
pos_encoder.pos_embedding.shape

torch.Size([1, 1025, 256])

In [9]:
embeddings = pos_encoder(embeddings)
embeddings.shape

torch.Size([128, 1025, 256])

# Scaled Dot Product Attention (SDPA)

In [10]:
dtype = torch.float32
query = torch.randn(batch_size, num_heads, sequence_length, embedding_dim // num_heads, device=device, dtype=dtype)
key = torch.randn(batch_size, num_heads, sequence_length, embedding_dim // num_heads, device=device, dtype=dtype)
value = torch.randn(batch_size, num_heads, sequence_length, embedding_dim // num_heads, device=device, dtype=dtype)


sdpa_output = F.scaled_dot_product_attention(query=query, key=key, value=value, attn_mask=None)
sdpa_output.shape

torch.Size([128, 32, 1025, 8])

# Multi Head Attention

In [11]:
query = torch.randn(batch_size, sequence_length, embedding_dim, device=device, dtype=dtype)
key = torch.randn(batch_size, sequence_length, embedding_dim, device=device, dtype=dtype)
value = torch.randn(batch_size, sequence_length, embedding_dim, device=device, dtype=dtype)
query.shape

torch.Size([128, 1025, 256])

In [12]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert embedding_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads
        self.qkv_projection = nn.Linear(in_features=3*embedding_dim, out_features=3*embedding_dim, bias=False)
        self.output_projection = nn.Linear(in_features=embedding_dim, out_features=embedding_dim, bias=False)
        self.output_dropout = nn.Dropout(p=dropout)
        self.dropout_p = dropout
    
    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, attn_mask: Optional[torch.Tensor]) -> torch.Tensor:
        qkv = torch.cat([query, key, value], dim=-1)
        qkv = self.qkv_projection(qkv)
        q, k, v = rearrange(qkv, 'b s (n h d) -> n b h s d', n=3, h=self.num_heads, d=self.head_dim).unbind(0)
        heads = F.scaled_dot_product_attention(query=q, key=k, value=v, attn_mask=attn_mask, dropout_p=self.dropout_p if self.training else 0.0)
        concatenated_heads = rearrange(heads, "b h s d_head -> b s (h d_head)")
        output = self.output_projection(concatenated_heads)
        output = self.output_dropout(output)
        return output

In [13]:
mha = MultiHeadAttention(embedding_dim=embedding_dim, num_heads=num_heads, dropout=dropout).to(device)

In [14]:
out = mha(query=query, key=key, value=value, attn_mask=None)
out.shape

torch.Size([128, 1025, 256])

# Feed Forward Network

In [15]:
class Mlp(nn.Module):
    def __init__(self, embedding_dim: int, d_ff: int, dropout: float = 0.0):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(in_features=embedding_dim, out_features=d_ff, bias=True),
            nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(in_features=d_ff, out_features=embedding_dim, bias=True),
            nn.Dropout(p=dropout),
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

In [16]:
mlp = Mlp(embedding_dim=embedding_dim, d_ff=2048, dropout=dropout).to(device)

out = mlp(out)
out.shape

torch.Size([128, 1025, 256])

# Encoder Layer

In [17]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, embedding_dim: int, num_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(normalized_shape=embedding_dim)
        self.mha = MultiHeadAttention(embedding_dim=embedding_dim, num_heads=num_heads, dropout=dropout)
        self.ln2 = nn.LayerNorm(normalized_shape=embedding_dim)
        self.mlp = Mlp(embedding_dim=embedding_dim, d_ff=d_ff, dropout=dropout)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mha_input = self.ln1(x)
        x = x + self.mha(query=mha_input, key=mha_input, value=mha_input, attn_mask=None)
        mlp_input = self.ln2(x)
        x = x + self.mlp(mlp_input)
        return x

In [18]:
encoder_layer = TransformerEncoderLayer(embedding_dim=embedding_dim, num_heads=num_heads, d_ff=2048, dropout=dropout).to(device)

layer_out = encoder_layer(embeddings)
layer_out.shape

torch.Size([128, 1025, 256])

# Encoder

In [19]:
class TransformerEncoder(nn.Module):
    def __init__(self, embedding_dim: int, num_heads: int, d_ff: int, num_layers: int, dropout: float):
        super().__init__()
        self.ln = nn.LayerNorm(normalized_shape=embedding_dim)
        self.layers = nn.Sequential(*[TransformerEncoderLayer(
            embedding_dim=embedding_dim,
            num_heads=num_heads,
            d_ff=d_ff,
            dropout=dropout
        ) for _ in range(num_layers)])
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.ln(self.layers(x)[:, 0])
        return y

In [20]:
encoder = TransformerEncoder(embedding_dim=embedding_dim, num_heads=num_heads, d_ff=2048, num_layers=2, dropout=dropout).to(device)

encoder_out = encoder(embeddings)
encoder_out.shape

torch.Size([128, 256])

# MLP Heads

In [21]:
class MlpPreTrainingHead(nn.Module):
    def __init__(self, embedding_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features=embedding_dim, out_features=hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, num_classes)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)

class MlpFineTuningHead(nn.Module):
    def __init__(self, embedding_dim: int, num_classes: int):
        super().__init__()
        self.layer = nn.Linear(in_features=embedding_dim, out_features=num_classes)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layer(x)

In [22]:
mlp_pretraining_head = MlpPreTrainingHead(embedding_dim=embedding_dim, hidden_dim=2048, num_classes=10).to(device)

y_out = mlp_pretraining_head(encoder_out)
y_out.shape

torch.Size([128, 10])

In [23]:
mlp_finetuning_head = MlpFineTuningHead(embedding_dim=embedding_dim, num_classes=10).to(device)

y_out2 = mlp_finetuning_head(encoder_out)
y_out2.shape

torch.Size([128, 10])

# ViT

In [24]:
class Vit(nn.Module):
    def __init__(
        self,
        image_embedder: ImageEmbedder,
        pos_encoder: Learned1dPositionEncoder,
        encoder: TransformerEncoder,
        head: nn.Module,
        dropout: float = 0.0
    ):
        super().__init__()
        self.image_embedder = image_embedder
        self.pos_encoder = pos_encoder
        self.dropout = nn.Dropout(p=dropout)
        self.encoder = encoder
        self.head = head
    
    def forward(self, image: torch.Tensor) -> torch.Tensor:
        embeddings = self.image_embedder(image)
        embeddings = self.pos_encoder(embeddings)
        embeddings = self.dropout(embeddings)
        cls_embedding = self.encoder(embeddings)
        y = self.head(cls_embedding)
        return y

In [25]:
vit = Vit(image_embedder=image_embedder,
          pos_encoder=pos_encoder,
          encoder=encoder,
          head=mlp_pretraining_head,
          dropout=dropout).to(device)

In [26]:
y = vit(image)
y.shape

torch.Size([128, 10])